# 05 — Temporal modelling: Model A attempt-level, Model B student-level

## Targets and evaluation units
At each eligible landmark (attempt 2 through the penultimate observed attempt), fit two classifiers:

- **Model A — near-term dropout-associated finality:** positive when the next attempt is the student's final observed attempt AND the supplied `target_dropout` label is 1. Evaluate Model A at the **attempt level**. Each landmark is a separate prediction decision.
- **Model B — eventual labelled dropout:** the supplied student-level `target_dropout` label repeated at each landmark. Evaluate Model B at the **student level** using an **any-landmark alert policy**: if the model predicts positive at any eligible landmark for a student, that student is predicted positive.

For Model B, threshold selection is also performed at student level: validation landmark probabilities are aggregated using the maximum probability per student, and the threshold maximizes student-level recall subject to the required student-level precision floor. Since `max(probability) >= threshold` is equivalent to at least one landmark crossing the threshold, this matches the operational policy.

The dataset has no dropout timestamp, so neither target establishes when dropout occurred. Model A's “final observed attempt” is bounded by the dataset observation window.

## Design
- Features are available after completing the current attempt and use history through that attempt only.
- Train/validation/test splits are by student ID; a student's rows never cross partitions.
- Select thresholds on validation data to maximize recall subject to precision floors of 40%, 50%, and 60%, using the task's evaluation unit.
- Model A reports attempt-level metrics only. Model B reports student-level any-landmark metrics as its primary and only headline evaluation.
- Compare unweighted logistic regression, class-weighted logistic regression, and class-weighted random forest.
- Students with one or two attempts are excluded from temporal modelling but retained in a separate descriptive outcome report.


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix
from IPython.display import display

ROOT_CANDIDATES = [Path("..").resolve(), Path("/mnt/data/student_success_assistant_v2").resolve(), Path(".").resolve()]
ROOT = next((p for p in ROOT_CANDIDATES if (p / "data" / "curated" / "temporal_attempt_prediction_rows.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate data/curated/temporal_attempt_prediction_rows.csv. Run Notebook 03 first.")
DATA = ROOT / "data" / "curated"
OUT = ROOT / "outputs" / "metrics"
OUT.mkdir(parents=True, exist_ok=True)
samples = pd.read_csv(DATA / "temporal_attempt_prediction_rows.csv", parse_dates=["attempt_dt", "signup_dt", "student_last_attempt_dt"])
eligibility = pd.read_csv(DATA / "temporal_eligibility_report.csv")
print("Project root:", ROOT)
print("Prediction rows:", len(samples), "| eligible students:", samples.student_id.nunique())
print("Model A positive rate:", round(samples.target_model_a_dropout_after_next_attempt.mean(), 3))
print("Model B positive rate:", round(samples.target_model_b_eventual_dropout.mean(), 3))

Project root: /Users/abishek/Documents/Projects/student_success_assistant
Prediction rows: 1476 | eligible students: 438
Model A positive rate: 0.054
Model B positive rate: 0.112


## 1. Feature contract and label audit

The feature list deliberately excludes `student_id`, timestamps, `observed_attempt_count`, `student_last_attempt_dt`, all target columns, and `next_attempt_is_final_observed`. The current attempt's score and duration are included as part of the available history through the current landmark. No future attempt fields are used.

In [2]:

TARGETS = {
    "model_a_near_term_dropout": "target_model_a_dropout_after_next_attempt",
    "model_b_eventual_dropout": "target_model_b_eventual_dropout",
}
META_COLS = {
    "student_id", "attempt_id", "attempt_dt", "signup_dt", "student_last_attempt_dt",
    "observed_attempt_count", "target_dropout", "next_attempt_is_final_observed",
    "target_model_a_dropout_after_next_attempt", "target_model_b_eventual_dropout",
}
all_predictor_cols = [c for c in samples.columns if c not in META_COLS]
assert not any("target" in c.lower() or c in {"student_id", "attempt_id", "student_last_attempt_dt", "observed_attempt_count"} for c in all_predictor_cols)
assert samples.groupby("student_id").sequence_index.min().eq(2).all()
assert samples.groupby("student_id").next_attempt_is_final_observed.sum().eq(1).all()
assert (samples[TARGETS["model_a_near_term_dropout"]] ==
        ((samples["next_attempt_is_final_observed"] == 1) & (samples["target_dropout"] == 1)).astype(int)).all()

# Explicit ablation: compare history-only, static-profile-only, and combined predictors.
STATIC_FEATURES = ["age_group", "country", "device", "signup_month", "signup_day_of_week", "signup_day_of_month"]
STATIC_FEATURES = [c for c in STATIC_FEATURES if c in all_predictor_cols]
TEMPORAL_FEATURES = [c for c in all_predictor_cols if c not in STATIC_FEATURES]
FEATURE_SETS = {
    "temporal_history_only": TEMPORAL_FEATURES,
    "student_static_only": STATIC_FEATURES,
    "combined_static_and_history": all_predictor_cols,
}
display(pd.DataFrame([
    {"feature_set": name, "feature_count": len(cols), "features": ", ".join(cols)}
    for name, cols in FEATURE_SETS.items()
]))
display(pd.DataFrame([
    {"task": name, "target": target, "positive_rows": int(samples[target].sum()),
     "rows": len(samples), "positive_rate": samples[target].mean()}
    for name, target in TARGETS.items()
]))


,feature_set,feature_count,features
0,temporal_history_only,27,"sequence_index, days_from_signup, gap_since_pr..."
1,student_static_only,6,"age_group, country, device, signup_month, sign..."
2,combined_static_and_history,33,"sequence_index, days_from_signup, gap_since_pr..."


,task,target,positive_rows,rows,positive_rate
0,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,80,1476,0.054201
1,model_b_eventual_dropout,target_model_b_eventual_dropout,166,1476,0.112466


## 2. Students with fewer than three attempts

These students cannot contribute a landmark at attempts 2 through penultimate. We report their supplied dropout-label distribution separately; this is descriptive and does not infer dropout solely from a low attempt count.

In [3]:
short_students = eligibility[eligibility["observed_attempt_count"] < 3].copy()
short_summary = (short_students.groupby(["observed_attempt_count", "target_dropout"], dropna=False)
                 .size().rename("students").reset_index())
short_summary["dropout_label"] = short_summary["target_dropout"].map({1: "yes", 0: "no"})
display(short_summary)
short_students.to_csv(OUT / "temporal_short_history_students.csv", index=False)
short_summary.to_csv(OUT / "temporal_short_history_outcome_summary.csv", index=False)

,observed_attempt_count,target_dropout,students,dropout_label
0,1,0,1,no
1,1,1,20,yes
2,2,0,16,no
3,2,1,24,yes


## 3. Feature ablation, preprocessing, model candidates, and metrics

We compare three predictor sets: temporal attempt history only, student-level/static attributes only, and their combination. Static attributes (age group, country, device, and signup calendar fields) are repeated across a student's eligible attempt rows. These are safe at prediction time. `refund_requested` is intentionally excluded from the primary models because its timestamp is unavailable and it may encode post-intervention or outcome-adjacent information.

In [4]:
def make_pipeline(df, model_kind):
    cat = [c for c in feature_cols if c in df.columns and
           (df[c].dtype == "object" or str(df[c].dtype).startswith("category"))]
    num = [c for c in feature_cols if c not in cat]
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat)
    ])
    if model_kind == "logistic_unweighted":
        clf = LogisticRegression(max_iter=3000, class_weight=None, random_state=42)
    elif model_kind == "logistic_balanced":
        clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
    elif model_kind == "random_forest_balanced":
        clf = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, class_weight="balanced",
                                     random_state=42, n_jobs=-1)
    else:
        raise ValueError(model_kind)
    return Pipeline([("prep", prep), ("model", clf)])

def binary_metrics(y, prob, threshold=0.5):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(prob) >= threshold
    cm = confusion_matrix(y, pred, labels=[0, 1])
    return {
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "PR_AUC": average_precision_score(y, prob) if len(np.unique(y)) > 1 else np.nan,
        "ROC_AUC": roc_auc_score(y, prob) if len(np.unique(y)) > 1 else np.nan,
        "TN": int(cm[0,0]), "FP": int(cm[0,1]), "FN": int(cm[1,0]), "TP": int(cm[1,1]),
        "flagged": int(pred.sum()), "n": len(y),
    }

def choose_threshold(y, prob, precision_floor):
    # Favor recall, then precision, then the higher threshold among ties.
    candidates = np.unique(np.r_[np.arange(0.05, 0.951, 0.01), 1.01])
    feasible = []
    for t in candidates:
        m = binary_metrics(y, prob, t)
        if m["precision"] >= precision_floor:
            feasible.append((m["recall"], m["precision"], t))
    if not feasible:
        return np.nan, False
    best = max(feasible, key=lambda x: (x[0], x[1], x[2]))
    return float(best[2]), True

def student_split(df, target, regime="random", seed=42):
    # Split using the student-level supplied dropout label, never the first attempt-row target.
    # Model A's early landmark rows are usually zero even for students who eventually drop out.
    ids = df.groupby("student_id", as_index=False).agg(target=("target_dropout", "first"),
                                                       first_landmark=("attempt_dt", "min"),
                                                       last_landmark=("student_last_attempt_dt", "max"))
    if regime == "random":
        tr_ids, rem_ids = train_test_split(ids.student_id, test_size=0.40, random_state=seed,
                                            stratify=ids["target"])
        rem = ids[ids.student_id.isin(rem_ids)]
        va_ids, te_ids = train_test_split(rem.student_id, test_size=0.50, random_state=seed,
                                           stratify=rem["target"])
        return set(tr_ids), set(va_ids), set(te_ids), {}
    if regime == "chronological":
        # Sort by first eligible landmark, not final attempt date, to avoid conditioning the split on future activity.
        ordered = ids.sort_values(["first_landmark", "student_id"]).reset_index(drop=True)
        n = len(ordered)
        a, b = int(0.60*n), int(0.80*n)
        return (set(ordered.iloc[:a].student_id), set(ordered.iloc[a:b].student_id),
                set(ordered.iloc[b:].student_id),
                {"train_cutoff": str(ordered.iloc[a-1].first_landmark.date()),
                 "validation_cutoff": str(ordered.iloc[b-1].first_landmark.date()),
                 "test_start": str(ordered.iloc[b].first_landmark.date())})
    raise ValueError(regime)

def row_weights(train_rows, target):
    # Each student contributes total weight 1 regardless of how many landmark rows they have.
    counts = train_rows.groupby("student_id")[target].transform("size").astype(float)
    return 1.0 / counts

## 4. Train, tune thresholds on validation, and evaluate on test

Model A threshold selection and metrics are attempt-level. Model B validation probabilities are first aggregated to one score per student (`max` across eligible landmarks), and its threshold is selected using student-level precision and recall. Test evaluation follows the same unit and any-landmark policy. The test set is never used for threshold selection.


In [5]:
all_results = []
all_predictions = []
all_split_rows = []

for task_name, target in TARGETS.items():
    evaluation_unit = "attempt" if task_name == "model_a_near_term_dropout" else "student_any_landmark"
    for regime in ["random", "chronological"]:
        tr_ids, va_ids, te_ids, split_meta = student_split(samples, target, regime=regime, seed=42)
        tr_all = samples[samples.student_id.isin(tr_ids)].copy()
        va_all = samples[samples.student_id.isin(va_ids)].copy()
        te_all = samples[samples.student_id.isin(te_ids)].copy()
        assert not (tr_ids & va_ids or tr_ids & te_ids or va_ids & te_ids)
        all_split_rows.append({"task": task_name, "evaluation_unit": evaluation_unit,
                               "regime": regime, "train_students": len(tr_ids),
                               "validation_students": len(va_ids), "test_students": len(te_ids),
                               "train_rows": len(tr_all), "validation_rows": len(va_all),
                               "test_rows": len(te_all), **split_meta})
        for feature_set_name, selected_features in FEATURE_SETS.items():
            feature_cols = selected_features
            for model_kind in ["logistic_unweighted", "logistic_balanced", "random_forest_balanced"]:
                model = make_pipeline(tr_all, model_kind)
                model.fit(tr_all[feature_cols], tr_all[target].astype(int),
                          model__sample_weight=row_weights(tr_all, target).to_numpy())
                pv_rows = model.predict_proba(va_all[feature_cols])[:, 1]
                pt_rows = model.predict_proba(te_all[feature_cols])[:, 1]

                if evaluation_unit == "attempt":
                    # Model A: each prediction landmark is an independent evaluation row.
                    yv = va_all[target].astype(int).to_numpy()
                    yt = te_all[target].astype(int).to_numpy()
                    pv, pt = pv_rows, pt_rows
                    val_frame = va_all[["student_id", target]].copy()
                    test_frame = te_all[["student_id", target]].copy()
                else:
                    # Model B: one score/label per student. Any landmark above threshold => positive.
                    val_frame = va_all[["student_id", target]].copy()
                    val_frame["probability"] = pv_rows
                    val_frame = val_frame.groupby("student_id", as_index=False).agg(
                        y=(target, "first"), probability=("probability", "max"))
                    test_frame = te_all[["student_id", target]].copy()
                    test_frame["probability"] = pt_rows
                    test_frame = test_frame.groupby("student_id", as_index=False).agg(
                        y=(target, "first"), probability=("probability", "max"))
                    yv = val_frame["y"].astype(int).to_numpy()
                    yt = test_frame["y"].astype(int).to_numpy()
                    pv = val_frame["probability"].to_numpy()
                    pt = test_frame["probability"].to_numpy()

                for floor in [0.40, 0.50, 0.60]:
                    threshold, feasible = choose_threshold(yv, pv, floor)
                    row = {"task": task_name, "target": target, "evaluation_unit": evaluation_unit,
                           "regime": regime, "model": model_kind, "feature_set": feature_set_name,
                           "precision_floor": floor, "threshold": threshold,
                           "threshold_feasible_on_validation": feasible}
                    if feasible:
                        test_m = binary_metrics(yt, pt, threshold)
                        val_m = binary_metrics(yv, pv, threshold)
                        row.update(test_m)
                        row["precision"] = test_m["precision"]
                        row["recall"] = test_m["recall"]
                        row["validation_precision"] = val_m["precision"]
                        row["validation_recall"] = val_m["recall"]
                        row["test_precision_floor_met"] = bool(test_m["precision"] >= floor)
                        all_results.append(row)

                        if evaluation_unit == "attempt":
                            # Include attempt metadata for Model A's per-landmark output.
                            meta = te_all[["student_id", "attempt_id", "attempt_dt", "sequence_index"]].copy()
                            meta["probability"] = pt_rows
                            meta["target"] = te_all[target].astype(int).to_numpy()
                            meta["predicted"] = (pt_rows >= threshold).astype(int)
                            meta["task"] = task_name
                            meta["evaluation_unit"] = evaluation_unit
                            meta["regime"] = regime
                            meta["model"] = model_kind
                            meta["feature_set"] = feature_set_name
                            meta["precision_floor"] = floor
                            meta["threshold"] = threshold
                            all_predictions.append(meta)
                        else:
                            pred_out = test_frame.copy().rename(columns={"y": "target"})
                            pred_out["predicted"] = (pred_out["probability"] >= threshold).astype(int)
                            pred_out["task"] = task_name
                            pred_out["evaluation_unit"] = evaluation_unit
                            pred_out["regime"] = regime
                            pred_out["model"] = model_kind
                            pred_out["feature_set"] = feature_set_name
                            pred_out["precision_floor"] = floor
                            pred_out["threshold"] = threshold
                            all_predictions.append(pred_out[["student_id", "target", "probability", "predicted",
                                                             "task", "evaluation_unit", "regime", "model",
                                                             "feature_set", "precision_floor", "threshold"]])
                    else:
                        row.update({"precision": np.nan, "recall": np.nan, "f1": np.nan,
                                    "PR_AUC": np.nan, "ROC_AUC": np.nan, "TN": np.nan, "FP": np.nan,
                                    "FN": np.nan, "TP": np.nan, "flagged": np.nan,
                                    "n": len(yv), "test_precision_floor_met": False,
                                    "note": "No validation threshold met precision floor"})
                        all_results.append(row)

results_df = pd.DataFrame(all_results)
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
split_df = pd.DataFrame(all_split_rows)

# Keep the output tables aligned with the project's intended primary metric and evaluation unit.
display(results_df.sort_values(["task", "regime", "feature_set", "precision_floor", "recall"],
                               ascending=[True, True, True, True, False]).head(60))
display(split_df)


,task,target,evaluation_unit,regime,model,feature_set,precision_floor,threshold,threshold_feasible_on_validation,precision,...,TN,FP,FN,TP,flagged,n,validation_precision,validation_recall,test_precision_floor_met,note
45,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,logistic_unweighted,combined_static_and_history,0.4,0.49,True,0.083333,...,249.0,11.0,9.0,1.0,12.0,270,0.421053,0.380952,False,NaN
48,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,logistic_balanced,combined_static_and_history,0.4,0.94,True,0.100000,...,251.0,9.0,9.0,1.0,10.0,270,0.470588,0.380952,False,NaN
51,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,random_forest_balanced,combined_static_and_history,0.4,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,281,NaN,NaN,False,No validation threshold met precision floor
46,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,logistic_unweighted,combined_static_and_history,0.5,0.72,True,0.000000,...,257.0,3.0,10.0,0.0,3.0,270,0.500000,0.142857,False,NaN
49,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,logistic_balanced,combined_static_and_history,0.5,0.95,True,0.000000,...,252.0,8.0,10.0,0.0,8.0,270,0.500000,0.333333,False,NaN
52,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,random_forest_balanced,combined_static_and_history,0.5,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,281,NaN,NaN,False,No validation threshold met precision floor
47,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,logistic_unweighted,combined_static_and_history,0.6,0.91,True,0.000000,...,259.0,1.0,10.0,0.0,1.0,270,1.000000,0.095238,False,NaN
50,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,logistic_balanced,combined_static_and_history,0.6,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,281,NaN,NaN,False,No validation threshold met precision floor
53,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,random_forest_balanced,combined_static_and_history,0.6,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,281,NaN,NaN,False,No validation threshold met precision floor
42,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,random_forest_balanced,student_static_only,0.4,0.59,True,0.000000,...,260.0,0.0,10.0,0.0,0.0,270,1.000000,0.047619,False,NaN


,task,evaluation_unit,regime,train_students,validation_students,test_students,train_rows,validation_rows,test_rows,train_cutoff,validation_cutoff,test_start
0,model_a_near_term_dropout,attempt,random,262,88,88,870,305,301,NaN,NaN,NaN
1,model_a_near_term_dropout,attempt,chronological,262,88,88,925,281,270,2026-07-22,2026-08-01,2026-08-01
2,model_b_eventual_dropout,student_any_landmark,random,262,88,88,870,305,301,NaN,NaN,NaN
3,model_b_eventual_dropout,student_any_landmark,chronological,262,88,88,925,281,270,2026-07-22,2026-08-01,2026-08-01


## 5. Save reports and check student-wise split integrity

The main comparison file contains Model A attempt-level results and Model B student-level any-landmark results. `test_precision_floor_met` explicitly indicates whether the held-out test precision remains above the selected validation precision floor; it is a diagnostic, not a tuning criterion.


In [6]:
results_df.to_csv(OUT / "temporal_primary_model_comparison.csv", index=False)
results_df[results_df.task.eq("model_a_near_term_dropout")].to_csv(
    OUT / "temporal_model_a_attempt_level_metrics.csv", index=False)
results_df[results_df.task.eq("model_b_eventual_dropout")].to_csv(
    OUT / "temporal_model_b_student_level_any_landmark_metrics.csv", index=False)
predictions_df.to_csv(OUT / "temporal_test_predictions_by_task_unit.csv", index=False)
split_df.to_csv(OUT / "temporal_student_split_report.csv", index=False)

# Explicitly report how many eligible students and short-history students are represented.
coverage = pd.DataFrame([
    {"group": "all_students_in_eligibility_report", "students": eligibility.student_id.nunique()},
    {"group": "students_with_1_or_2_attempts", "students": int((eligibility.observed_attempt_count < 3).sum())},
    {"group": "students_eligible_for_temporal_modeling", "students": int((eligibility.observed_attempt_count >= 3).sum())},
    {"group": "students_with_prediction_rows", "students": samples.student_id.nunique()},
])
display(coverage)
coverage.to_csv(OUT / "temporal_eligibility_coverage.csv", index=False)

# Guardrails: Model A must be attempt-level; Model B must be student-level.
assert results_df.loc[results_df.task.eq("model_a_near_term_dropout"), "evaluation_unit"].eq("attempt").all()
assert results_df.loc[results_df.task.eq("model_b_eventual_dropout"), "evaluation_unit"].eq("student_any_landmark").all()
print("Saved temporal model reports to:", OUT)


,group,students
0,all_students_in_eligibility_report,499
1,students_with_1_or_2_attempts,61
2,students_eligible_for_temporal_modeling,438
3,students_with_prediction_rows,438


Saved temporal model reports to: /Users/abishek/Documents/Projects/student_success_assistant/outputs/metrics


## 6. Interpretation and limitations

- **Model A** is evaluated at attempt level: every eligible landmark is a separate prediction decision. Its target means the next attempt is the final observed attempt and the supplied dropout label is positive; it does not establish actual dropout timing.
- **Model B** is evaluated at student level using an any-landmark policy. A student is predicted positive if any eligible landmark probability reaches the threshold. Threshold tuning is done on validation students using the maximum landmark probability per student, so the precision constraint and recall objective match the deployed unit.
- The student-level `max` score is a ranking summary for the any-landmark policy, not a claim that the maximum is available at the earliest landmark. The policy means the student is flagged once any prediction crosses threshold over their eligible observed sequence.
- The chronological split orders students by their first eligible landmark. It is an extrapolation proxy, not a fully prospective event-time evaluation.
- Thresholds are selected on validation data; with a small dataset, precision estimates and threshold feasibility may be unstable. Test precision-floor status is reported separately and must not be used to retune the threshold.
- The supplied data has no dropout timestamp, so Model B predicts the supplied eventual label, not a verified dropout event within a defined future horizon.
